In [1]:
desired_version = "3.6.0"
version = None

try:
    import datasets
    version = datasets.__version__
except ImportError:
    pass

if version != desired_version:
    print(f"Installing datasets=={desired_version} ...")
    %pip install datasets=={desired_version}
else:
    print(f"Datasets is already at version {desired_version}")

Datasets is already at version 3.6.0


In [2]:
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
import pprint

# Dataset QA

**Link here:** https://huggingface.co/datasets?sort=trending&search=babi_qa

In [3]:
dataset = load_dataset("facebook/babi_qa", name="en-qa1")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['story'],
        num_rows: 200
    })
    test: Dataset({
        features: ['story'],
        num_rows: 200
    })
})

In [4]:
sample_0 = dataset["train"][0]
pprint.pprint(sample_0)

{'story': {'answer': ['',
                      '',
                      'bathroom',
                      '',
                      '',
                      'hallway',
                      '',
                      '',
                      'hallway',
                      '',
                      '',
                      'office',
                      '',
                      '',
                      'bathroom'],
           'id': ['1',
                  '2',
                  '3',
                  '4',
                  '5',
                  '6',
                  '7',
                  '8',
                  '9',
                  '10',
                  '11',
                  '12',
                  '13',
                  '14',
                  '15'],
           'supporting_ids': [[],
                              [],
                              ['1'],
                              [],
                              [],
                              ['4'],
           

In [5]:
# avoid type sample['story']
dataset_flattened = dataset.flatten()
pprint.pprint(dataset_flattened['train'][0])

{'story.answer': ['',
                  '',
                  'bathroom',
                  '',
                  '',
                  'hallway',
                  '',
                  '',
                  'hallway',
                  '',
                  '',
                  'office',
                  '',
                  '',
                  'bathroom'],
 'story.id': ['1',
              '2',
              '3',
              '4',
              '5',
              '6',
              '7',
              '8',
              '9',
              '10',
              '11',
              '12',
              '13',
              '14',
              '15'],
 'story.supporting_ids': [[],
                          [],
                          ['1'],
                          [],
                          [],
                          ['4'],
                          [],
                          [],
                          ['4'],
                          [],
                          [],
  

# Preprocessing

In [6]:
def get_QAs_each_sample(sample):
    context = []
    qas = []

    for i, type in enumerate(sample['story.type']):
        # if context
        if type == 0:
            context.append(sample['story.text'][i])
        # if question
        elif type == 1:
            qa_dict = {
                'context': ' '.join(context),
                'question': sample['story.text'][i],
                'answer': sample['story.answer'][i]
            }
            qas.append(qa_dict)

    return qas

In [7]:
get_QAs_each_sample(dataset_flattened['train'][0])

[{'context': 'Mary moved to the bathroom. John went to the hallway.',
  'question': 'Where is Mary?',
  'answer': 'bathroom'},
 {'context': 'Mary moved to the bathroom. John went to the hallway. Daniel went back to the hallway. Sandra moved to the garden.',
  'question': 'Where is Daniel?',
  'answer': 'hallway'},
 {'context': 'Mary moved to the bathroom. John went to the hallway. Daniel went back to the hallway. Sandra moved to the garden. John moved to the office. Sandra journeyed to the bathroom.',
  'question': 'Where is Daniel?',
  'answer': 'hallway'},
 {'context': 'Mary moved to the bathroom. John went to the hallway. Daniel went back to the hallway. Sandra moved to the garden. John moved to the office. Sandra journeyed to the bathroom. Mary moved to the hallway. Daniel travelled to the office.',
  'question': 'Where is Daniel?',
  'answer': 'office'},
 {'context': 'Mary moved to the bathroom. John went to the hallway. Daniel went back to the hallway. Sandra moved to the garden.

In [8]:
def process_QA_dataset(raw_dataset, func):
    all_qas = []
    for sample in raw_dataset:
        all_qas.extend(func(sample))

    dataset_processed = Dataset.from_list(all_qas)
    return dataset_processed

In [9]:
dataset_processed = process_QA_dataset(dataset_flattened['train'], get_QAs_each_sample)
dataset_processed

Dataset({
    features: ['context', 'question', 'answer'],
    num_rows: 1000
})

In [10]:
dataset_processed[1]

{'context': 'Mary moved to the bathroom. John went to the hallway. Daniel went back to the hallway. Sandra moved to the garden.',
 'question': 'Where is Daniel?',
 'answer': 'hallway'}

In [11]:
def get_answer_range(sample):
    start_idx = sample['context'].find(sample['answer'])
    end_idx = start_idx + len(sample['answer'])
    return {
        'start_idx': start_idx,
        'end_idx': end_idx
    }

In [12]:
dataset_processed_2 = dataset_processed.map(get_answer_range)
dataset_processed_2

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'question', 'answer', 'start_idx', 'end_idx'],
    num_rows: 1000
})

In [13]:
dataset_processed_2[0]

{'context': 'Mary moved to the bathroom. John went to the hallway.',
 'question': 'Where is Mary?',
 'answer': 'bathroom',
 'start_idx': 18,
 'end_idx': 26}

# Tokenizer

In [14]:
from transformers import DistilBertTokenizerFast, TFDistilBertForQuestionAnswering, DistilBertForQuestionAnswering
from transformers import Trainer, TrainingArguments

In [15]:
MODEL_NAME= "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
tf_model = TFDistilBertForQuestionAnswering.from_pretrained(MODEL_NAME, from_pt=True, return_dict=True)
tf_model.distilbert.trainable = False
# return output as dict instead of tuple

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForQuestionAnswering: ['vocab_projector.weight', 'vocab_layer_norm.weight', 'vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_projector.bias', 'vocab_transform.bias']
- This IS expected if you are initializing TFDistilBertForQuestionAnswering from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForQuestionAnswering from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForQuestionAnswering 

In [16]:
tf_model.summary()

Model: "tf_distil_bert_for_question_answering"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 distilbert (TFDistilBertMa  multiple                  66362880  
 inLayer)                                                        
                                                                 
 qa_outputs (Dense)          multiple                  1538      
                                                                 
 dropout_19 (Dropout)        multiple                  0         
                                                                 
Total params: 66364418 (253.16 MB)
Trainable params: 1538 (6.01 KB)
Non-trainable params: 66362880 (253.15 MB)
_________________________________________________________________


In [17]:
dropout_layer = tf_model.get_layer("dropout_19")
dropout_layer.rate = 0.0
print(dropout_layer.rate)

0.0


In [18]:
for w in tf_model.weights:
    print(f"{w.name:<100}: {w.shape}")

tf_distil_bert_for_question_answering/distilbert/embeddings/word_embeddings/weight:0                : (30522, 768)
tf_distil_bert_for_question_answering/distilbert/embeddings/position_embeddings/embeddings:0        : (512, 768)
tf_distil_bert_for_question_answering/distilbert/embeddings/LayerNorm/gamma:0                       : (768,)
tf_distil_bert_for_question_answering/distilbert/embeddings/LayerNorm/beta:0                        : (768,)
tf_distil_bert_for_question_answering/distilbert/transformer/layer_._0/attention/q_lin/kernel:0     : (768, 768)
tf_distil_bert_for_question_answering/distilbert/transformer/layer_._0/attention/q_lin/bias:0       : (768,)
tf_distil_bert_for_question_answering/distilbert/transformer/layer_._0/attention/k_lin/kernel:0     : (768, 768)
tf_distil_bert_for_question_answering/distilbert/transformer/layer_._0/attention/k_lin/bias:0       : (768,)
tf_distil_bert_for_question_answering/distilbert/transformer/layer_._0/attention/v_lin/kernel:0     : (768, 76

In [19]:
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [20]:
input_tokenized = tokenizer(
    dataset_processed_2['question'],
    dataset_processed_2['context'],
    max_length=tokenizer.model_max_length,
    padding='max_length',
    truncation=True,
    return_tensors="pt"
)

input_tokenized[0]

Encoding(num_tokens=512, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [21]:
def print_tokenized_sample(sample):
    fields = {
        "tokens": sample.tokens,
        "ids": sample.ids,
        "type_ids": sample.type_ids,
        "offsets": sample.offsets,
        "attention_mask": sample.attention_mask,
        "special_tokens_mask": sample.special_tokens_mask
    }

    tokens = fields["tokens"]
    pad_index = tokens.index("[PAD]") + 1 if "[PAD]" in tokens else len(tokens)
    # remove padding, keep 1 pad token
    truncated_fields = {k: v[:pad_index] for k, v in fields.items()}

    print(f"{'idx':<4} {'token':<15} {'id':<8} {'type':<6} {'attn':<6} {'spec':<6} {'offset':<10}")
    print("-" * 80)
    for i in range(pad_index):
        print(f"{i:<4} "
              f"{truncated_fields['tokens'][i]:<15} "
              f"{truncated_fields['ids'][i]:<8} "
              f"{truncated_fields['type_ids'][i]:<6} "
              f"{truncated_fields['attention_mask'][i]:<6} "
              f"{truncated_fields['special_tokens_mask'][i]:<6} "
              f"{str(tuple(truncated_fields['offsets'][i])):<10}")

    print("-" * 80)
    print(sample.overflowing)

In [22]:
print_tokenized_sample(input_tokenized[0])

idx  token           id       type   attn   spec   offset    
--------------------------------------------------------------------------------
0    [CLS]           101      0      1      1      (0, 0)    
1    where           2073     0      1      0      (0, 5)    
2    is              2003     0      1      0      (6, 8)    
3    mary            2984     0      1      0      (9, 13)   
4    ?               1029     0      1      0      (13, 14)  
5    [SEP]           102      0      1      1      (0, 0)    
6    mary            2984     1      1      0      (0, 4)    
7    moved           2333     1      1      0      (5, 10)   
8    to              2000     1      1      0      (11, 13)  
9    the             1996     1      1      0      (14, 17)  
10   bathroom        5723     1      1      0      (18, 26)  
11   .               1012     1      1      0      (26, 27)  
12   john            2198     1      1      0      (28, 32)  
13   went            2253     1      1      0      

In [23]:
# character (not word) position -> token position
print(input_tokenized[0].char_to_token(13, sequence_index=0))
print(input_tokenized[0].char_to_token(13, sequence_index=1))

4
None


In [24]:
def tokenize_and_assign_labels(sample, tokenizer):
    tokenized_sample = tokenizer(
        sample['question'],
        sample['context'],
        max_length=tokenizer.model_max_length,
        padding='max_length',
        truncation=True,
        return_tensors="pt"
    )  # return shape of (1, max_length)

    # squeeze to shape (max_length, )
    input_ids = tokenized_sample["input_ids"].squeeze(0)
    attention_mask = tokenized_sample["attention_mask"].squeeze(0)

    start_token_idx = tokenized_sample.char_to_token(sample['start_idx'], sequence_index=1)
    end_token_idx = tokenized_sample.char_to_token(sample['end_idx'] - 1, sequence_index=1)
    # to get the full answer, get context[start_idx, end_idx), so we -1 back

    if start_token_idx is None or end_token_idx is None:
        return None

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'start_token_idx': start_token_idx,
        'end_token_idx': end_token_idx
    }

In [25]:
dataset_tokenized = dataset_processed_2.map(
    lambda x: tokenize_and_assign_labels(x, tokenizer)
)

dataset_tokenized

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['context', 'question', 'answer', 'start_idx', 'end_idx', 'input_ids', 'attention_mask', 'start_token_idx', 'end_token_idx'],
    num_rows: 1000
})

In [26]:
np.array(dataset_tokenized['input_ids']).shape

(1000, 512)

In [27]:
np.array(dataset_tokenized['start_token_idx']).shape

(1000,)

# Processing Pipeline

In [28]:
import tensorflow as tf
from tensorflow import keras
import time

In [29]:
def go_preprocessing_pipeline(raw_dataset, tokenizer):
    dataset = raw_dataset.flatten()

    train_dataset = process_QA_dataset(dataset['train'], get_QAs_each_sample)
    test_dataset = process_QA_dataset(dataset['test'], get_QAs_each_sample)

    dataset = DatasetDict({
        'train': train_dataset,
        'test': test_dataset
    })

    dataset = dataset.map(get_answer_range, desc="get_answer_range")
    dataset = dataset.map(lambda x: tokenize_and_assign_labels(x, tokenizer), desc="tokenize_and_assign_labels")

    dataset['train'] = dataset['train'].filter(lambda x: x is not None)
    dataset['test']  = dataset['test'].filter(lambda x: x is not None)

    return dataset

In [30]:
final_dataset = go_preprocessing_pipeline(dataset, tokenizer)
final_dataset

get_answer_range:   0%|          | 0/1000 [00:00<?, ? examples/s]

get_answer_range:   0%|          | 0/1000 [00:00<?, ? examples/s]

tokenize_and_assign_labels:   0%|          | 0/1000 [00:00<?, ? examples/s]

tokenize_and_assign_labels:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'answer', 'start_idx', 'end_idx', 'input_ids', 'attention_mask', 'start_token_idx', 'end_token_idx'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['context', 'question', 'answer', 'start_idx', 'end_idx', 'input_ids', 'attention_mask', 'start_token_idx', 'end_token_idx'],
        num_rows: 1000
    })
})

In [31]:
def convert_to_tf_dataset(hf_dataset, batch_size=32, shuffle=False):
    features = {
        "input_ids": tf.convert_to_tensor(hf_dataset["input_ids"], dtype=tf.int32),
        "attention_mask": tf.convert_to_tensor(hf_dataset["attention_mask"], dtype=tf.int32),
    }

    # reshape to compute loss easily
    labels = {
        "start_token_idx": tf.reshape(tf.convert_to_tensor(hf_dataset["start_token_idx"], dtype=tf.int32), [-1, ]),
        "end_token_idx": tf.reshape(tf.convert_to_tensor(hf_dataset["end_token_idx"], dtype=tf.int32), [-1, ]),
    }

    ds = tf.data.Dataset.from_tensor_slices((features, labels))

    if shuffle:
        ds = ds.shuffle(1000).repeat()

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [32]:
train_dataset = convert_to_tf_dataset(final_dataset["train"], batch_size=8, shuffle=False)
test_dataset  = convert_to_tf_dataset(final_dataset["test"], batch_size=8)

train_dataset, test_dataset

(<_PrefetchDataset element_spec=({'input_ids': TensorSpec(shape=(None, 512), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(None, 512), dtype=tf.int32, name=None)}, {'start_token_idx': TensorSpec(shape=(None,), dtype=tf.int32, name=None), 'end_token_idx': TensorSpec(shape=(None,), dtype=tf.int32, name=None)})>,
 <_PrefetchDataset element_spec=({'input_ids': TensorSpec(shape=(None, 512), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(None, 512), dtype=tf.int32, name=None)}, {'start_token_idx': TensorSpec(shape=(None,), dtype=tf.int32, name=None), 'end_token_idx': TensorSpec(shape=(None,), dtype=tf.int32, name=None)})>)

# Training

In [33]:
def compute_metrics(model, dataset, loss_fn=None):
    total_loss = 0.0
    n_batches = 0

    total_em = 0.0
    n_samples = 0

    for x_batch, y_batch in dataset:
        outputs = model(x_batch)
        start_logits, end_logits = outputs['start_logits'], outputs['end_logits']  # shape of (batch_size, seq_length)

        # Loss (optional)
        if loss_fn is not None:
            loss_start = loss_fn(y_batch['start_token_idx'], start_logits)
            loss_end   = loss_fn(y_batch['end_token_idx'], end_logits)
            loss_per_batch = (loss_start + loss_end) / 2

            total_loss += loss_per_batch
            n_batches += 1

        # EM
        start_pred = tf.argmax(start_logits, axis=1)  # shape of (batch_size, )
        end_pred = tf.argmax(end_logits, axis=1)

        start_true = tf.cast(y_batch['start_token_idx'], tf.int64)
        end_true   = tf.cast(y_batch['end_token_idx'], tf.int64)

        em_per_batch = tf.reduce_sum(
            tf.cast(
                tf.logical_and(start_pred == start_true, end_pred == end_true),
                tf.float32
            )
        )  # returns tensor scalar

        total_em += em_per_batch
        n_samples += x_batch["input_ids"].shape[0]


    avg_loss = (total_loss / n_batches).numpy() if loss_fn is not None else None
    avg_em = (total_em / n_samples).numpy()

    return avg_loss, avg_em

In [34]:
def fit_tf_model(tf_model, train_dataset, val_dataset, n_epochs=10, learning_rate=1e-4, patience=3, min_delta=1e-3):
    opt = keras.optimizers.Adam(learning_rate=learning_rate)
    loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    train_losses = []
    train_em_accuracies = []
    val_losses = []
    val_em_accuracies = []

    best_val_em = -1.0
    best_weights = None
    best_epoch = 0
    wait = 0

    for epoch in range(n_epochs):
        start_time = time.time()

        total_loss_per_epoch = 0
        n_batches = 0

        total_em_per_epoch = 0
        n_samples = 0

        for step, (x_batch, y_batch) in enumerate(train_dataset):
            with tf.GradientTape() as tape:
                outputs = tf_model(x_batch)
                start_logits, end_logits = outputs['start_logits'], outputs['end_logits']  # shape of (batch_size, seq_length)

                loss_start = loss_fn(y_batch['start_token_idx'], start_logits)
                loss_end   = loss_fn(y_batch['end_token_idx'], end_logits)
                loss_per_batch = (loss_start + loss_end) / 2

            grads = tape.gradient(loss_per_batch, tf_model.trainable_variables)
            opt.apply_gradients(zip(grads, tf_model.trainable_variables))

        train_loss, train_em = compute_metrics(tf_model, train_dataset, loss_fn)
        val_loss, val_em = compute_metrics(tf_model, val_dataset, loss_fn)

        train_losses.append(train_loss)
        train_em_accuracies.append(train_em)
        val_losses.append(val_loss)
        val_em_accuracies.append(val_em)

        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1} ({epoch_time:.2f}s) - Train Loss: {train_loss:.4f} | Train EM: {train_em:.4f} | Val Loss: {val_loss:.4f} | Val EM: {val_em:.4f}")

        # Early stopping
        if val_em > best_val_em + min_delta:
            best_val_em = val_em
            best_weights = tf_model.get_weights()
            best_epoch = epoch + 1
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch+1}. Best Val EM: {best_val_em:.4f}")
                break

    if best_weights is not None:
        print(f"Restoring best model weights from epoch {best_epoch}...")
        tf_model.set_weights(best_weights)

    return train_losses, train_em_accuracies, val_losses, val_em_accuracies

In [35]:
train_losses, train_em_accuracies, val_losses, val_em_accuracies = fit_tf_model(
    tf_model, train_dataset, test_dataset, n_epochs=20, learning_rate=2e-3
)

Epoch 1 (138.87s) - Train Loss: 1.4212 | Train EM: 0.4180 | Val Loss: 1.4663 | Val EM: 0.3960
Epoch 2 (119.62s) - Train Loss: 1.2678 | Train EM: 0.4640 | Val Loss: 1.3226 | Val EM: 0.4260
Epoch 3 (107.67s) - Train Loss: 1.2003 | Train EM: 0.4840 | Val Loss: 1.2620 | Val EM: 0.4500
Epoch 4 (107.15s) - Train Loss: 1.1640 | Train EM: 0.4980 | Val Loss: 1.2309 | Val EM: 0.4550
Epoch 5 (107.24s) - Train Loss: 1.1413 | Train EM: 0.5130 | Val Loss: 1.2125 | Val EM: 0.4630
Epoch 6 (107.12s) - Train Loss: 1.1253 | Train EM: 0.5200 | Val Loss: 1.2004 | Val EM: 0.4680
Epoch 7 (125.90s) - Train Loss: 1.1131 | Train EM: 0.5280 | Val Loss: 1.1918 | Val EM: 0.4740
Epoch 8 (107.92s) - Train Loss: 1.1031 | Train EM: 0.5280 | Val Loss: 1.1852 | Val EM: 0.4790
Epoch 9 (107.33s) - Train Loss: 1.0947 | Train EM: 0.5340 | Val Loss: 1.1799 | Val EM: 0.4870
Epoch 10 (107.02s) - Train Loss: 1.0873 | Train EM: 0.5390 | Val Loss: 1.1755 | Val EM: 0.4890
Epoch 11 (107.00s) - Train Loss: 1.0806 | Train EM: 0.5410 

# Inference

In [36]:
def answer_questions(questions, context, tokenizer, model):
    inputs = tokenizer(
        questions, [context]*len(questions), return_tensors="tf",
        max_length=512, padding="max_length", truncation=True
    )
    outputs = model(inputs)
    start_logits = outputs[0]  # shape of (batch_size, seq_length)
    end_logits = outputs[1]

    answers = []
    input_ids = inputs["input_ids"]  # shape of (batch_size, seq_length)

    for i in range(len(questions)):
        start_token_idx = tf.argmax(start_logits[i]).numpy()
        end_token_idx = tf.argmax(end_logits[i]).numpy()

        tokens = tokenizer.convert_ids_to_tokens(input_ids[i].numpy())
        answer_tokens = tokens[start_token_idx:end_token_idx+1]

        answer = tokenizer.convert_tokens_to_string(answer_tokens)
        answers.append(answer)

    return answers

In [37]:
context = "The apple is on the table. The orange is in the basket."

questions = [
    "What is on the table?",
    "Where is the orange?",
]

answers = answer_questions(questions, context, tokenizer, tf_model)
for q, a in zip(questions, answers):
    print(f"Q: {q:<60} -> A: {a}")

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Q: What is on the table?                                        -> A: table
Q: Where is the orange?                                         -> A: basket
